# Load silver dimensions

Loads `silver.dim_market`, `silver.dim_commodity` and `silver.dim_date` in the workspace's default catalog from the
rows `src/load_bronze_dims` loaded for the given run (`bronze.raw_dim_*`; the tables are created by `src/setup`).

- Values are cast to the silver types and checked first (`lib_mandi.dq_checks`): filled, castable, keys unique,
  no duplicate natural keys, no missing days in `dim_date`. If a table's rows fail a check, nothing is written to it.
- `updated_at` is set to the load time.
- INSERT OVERWRITE: each table is replaced by the run's rows. Add a dimension row by editing the CSV and running
  `src/load_bronze_dims` and this notebook again.

Parameters:
- `run_id` (required): the run whose bronze seed rows are loaded.
- `fail_on_error` (optional, `true`/`false`): raise at the end if a table was not loaded, so a job task fails.

If a table fails, the reason is printed and the notebook still completes unless `fail_on_error` is `true`.

In [ ]:
dbutils.widgets.text("run_id", "", "Run id")
dbutils.widgets.text("fail_on_error", "false", "Fail the job on error (true/false)")

In [ ]:
from lib_mandi.config import (
    RULES_DIM_COMMODITY,
    RULES_DIM_DATE,
    RULES_DIM_MARKET,
    SCHEMA_DIM_COMMODITY,
    SCHEMA_DIM_DATE,
    SCHEMA_DIM_MARKET,
)
from lib_mandi.extract import extract__silver_dims
from lib_mandi.load import load__silver_dims
from lib_mandi.transformations import transform__silver_dims
from lib_mandi.utility import get__default_catalog

run_id = dbutils.widgets.get("run_id").strip()
if not run_id:
    raise ValueError("Widget 'run_id' is required")

catalog = get__default_catalog(spark)
print(f"Loading silver dimensions in catalog: {catalog}")

if not catalog:
    raise RuntimeError("Could not read the default catalog (see the error printed above)")

dims = [
    (f"{catalog}.bronze.raw_dim_market", f"{catalog}.silver.dim_market", SCHEMA_DIM_MARKET, RULES_DIM_MARKET),
    (f"{catalog}.bronze.raw_dim_commodity", f"{catalog}.silver.dim_commodity", SCHEMA_DIM_COMMODITY, RULES_DIM_COMMODITY),
    (f"{catalog}.bronze.raw_dim_date", f"{catalog}.silver.dim_date", SCHEMA_DIM_DATE, RULES_DIM_DATE),
]

loaded = True
for bronze_table, silver_table, schema, rules in dims:
    df = extract__silver_dims(spark, run_id, bronze_table)
    if df is not None:
        df = transform__silver_dims(
            df, schema, rules["not_null_columns"], rules["unique_columns"], rules["date_column"]
        )
    if df is not None:
        loaded = load__silver_dims(df, silver_table) and loaded
    else:
        loaded = False

if not loaded and dbutils.widgets.get("fail_on_error").strip().lower() == "true":
    raise RuntimeError("Load did not finish (see the reason printed above)")